In [5]:
from pathlib import Path
import pandas as pd
import json
import os

ModuleNotFoundError: No module named 'fastparquet'

In [ ]:
top_path = Path(os.path.dirname(os.getcwd()))
data_path = top_path / "data"
insights_path = top_path / "reports" / "insights"

notebooks_path = top_path / "notebooks"
team_data_path = data_path / "processed" / "team_data.parquet"
predictions_path = insights_path / "OutcomePrediction_predictions.parquet"

In [ ]:
predictions_df = pd.read_parquet(predictions_path, engine='fastparquet').rename(columns={"result": "prediction"})
team_df = pd.read_parquet(team_data_path, engine='fastparquet')

ImportError: Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

## Store the accuracy per league in a dictionary

In [ ]:
league_data = team_df[["gameid", "side", "league", "result"]]
predictions_df = predictions_df.merge(league_data, on=["gameid", "side"], how="inner", validate="many_to_many")

In [ ]:
# Explore accuracy by league
leagues = predictions_df["league"].unique()
accuracies = {}

for league in leagues:
    league_df = predictions_df[predictions_df["league"] == league]
    accuracy = league_df["prediction"] == league_df["result"]
    accuracies[league] = {"count": len(league_df), "accuracy": accuracy.mean()}

accuracies = {k: v for k, v in sorted(accuracies.items(), key=lambda item: item[1]["accuracy"], reverse=True)}

with open(insights_path / "OutcomePrediction_league_accuracies.json", "w") as f:
    json.dump(accuracies, f)

# Specific League Predictions Analysis

In [ ]:
# Specific League Analysis
analysis_league = "LEC"
games_data =  team_df[["date", "gameid", "teamname", "opponentteam", "side"]]

lec_df = predictions_df[predictions_df["league"] == analysis_league][["gameid", "side", "league", "prediction", "result"]]
lec_df = games_data.merge(lec_df, on=["gameid", "side"])

lec_df["correct"] = lec_df["prediction"] == lec_df["result"]
lec_df.sort_values(by="date", inplace=True)

lec_df

,date,gameid,teamname,opponentteam,side,league,prediction,result,correct
0,2022-01-14 17:28:42,ESPORTSTMNT04_2090342,Rogue,SK Gaming,Blue,LEC,1,1,True
1,2022-01-14 17:28:42,ESPORTSTMNT04_2090342,SK Gaming,Rogue,Red,LEC,0,0,True
2,2022-01-14 19:14:41,ESPORTSTMNT04_2090354,Astralis,Misfits Gaming,Blue,LEC,0,0,True
3,2022-01-14 19:14:41,ESPORTSTMNT04_2090354,Misfits Gaming,Astralis,Red,LEC,1,1,True
4,2022-01-16 18:53:12,ESPORTSTMNT01_2692441,MAD Lions,G2 Esports,Blue,LEC,0,1,False
...,...,...,...,...,...,...,...,...,...
281,2024-06-23 15:08:04,LOLTMNT05_53011,Team Heretics,Rogue,Red,LEC,1,0,False
282,2024-06-30 17:00:33,LOLTMNT05_57516,Team BDS,Fnatic,Blue,LEC,1,1,True
283,2024-06-30 17:00:33,LOLTMNT05_57516,Fnatic,Team BDS,Red,LEC,0,0,True
284,2024-06-30 18:51:07,LOLTMNT05_59739,GiantX,Karmine Corp,Blue,LEC,1,0,False
